# ⚖️ CUAD Legal RAG Diagnostics & Evaluation Suite

This notebook guide helps you run modular RAG benchmarking experiments on the **CUAD (Contract Understanding Atticus Dataset)** and compiles a rich, 3-level diagnostic dashboard analyzing dataset profile, retrieval boundary splits, LLM failure modes, and latency breakdowns.

## 📂 Step 1: Set Up Project Workspace

Mount your Google Drive and unzip the workspace archive directly into the project folder:

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Navigate to workspace
%cd /content/drive/MyDrive/ragstack

# Unzip workspace archive to fetch latest diagnostic code updates
!unzip -o ragstack_colab.zip -d .

## 📦 Step 2: Install Dependencies

In [ ]:
!pip install -r requirements.txt

## 🦙 Step 3: Initialize Ollama (Optional for local 7B/8B GPU Models)

If you plan to run local GPU models (`llama3:8b`, `qwen2.5:7b`), run the cell below to install and deploy the Ollama daemon:

In [ ]:
# 1. Download and install Ollama
!apt-get update && apt-get install -y zstd pciutils lshw
!curl -fsSL https://ollama.com/install.sh | sh

# 2. Start the Ollama background daemon
import subprocess
import time
print("Starting Ollama server...")
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)  # Wait for server to start

# 3. Pull required models
print("Pulling llama3:8b...")
!ollama pull llama3:8b

print("Pulling qwen2.5:7b...")
!ollama pull qwen2.5:7b

## ⚙️ Step 4: Run RAG Pipeline Benchmark

Configure and run the benchmark on the **`cuad`** dataset using the interactive form variables below:

In [ ]:
#@title CUAD Experiment Configuration { run: "auto" }

n_records = 20 #@param {type:"integer"} # Slice size for testing (0 for complete dataset)
username = "colab_user" #@param {type:"string"}
chunker = "recursive" #@param ["recursive", "semantic", "metadata"]
chunk_size = 512 #@param {type:"integer"}
retriever = "hybrid" #@param ["dense", "sparse", "hybrid"]
embedder = "bge-small" #@param ["bge-small", "bge-large", "minilm", "mock"]
generator = "hf-small" #@param ["hf-small", "hf-large", "llama3:8b", "qwen2.5:7b", "mock"]
evaluator = "llm" #@param ["llm", "heuristic", "mock"]
run_id_prefix = "cuad_eval_run" #@param {type:"string"}

n_records_flag = f"-n {n_records}" if n_records > 0 else ""

!python main.py \
  --profile colab-gpu \
  --dataset cuad \
  {n_records_flag} \
  --username {username} \
  --chunker {chunker} \
  --chunk_size {chunk_size} \
  --retriever {retriever} \
  --embedder {embedder} \
  --generator {generator} \
  --evaluator {evaluator} \
  --run_id_prefix {run_id_prefix}


## 🤖 Step 4.5: Run Automated Self-Tuning Optimizer

Instead of manual trial-and-error, run the coordinate-descent self-tuning optimizer. It sweeps each parameter sequentially (locking in the best choice before moving to the next):

In [ ]:
#@title RAG Auto-Tuning Configuration { run: "auto" }

n_records_tune = 10 #@param {type:"integer"} # Records to use for each tuning iteration
metric_to_optimize = "mean_adherence" #@param ["mean_adherence", "mean_context_relevance", "mean_context_utilization", "mean_completeness"]

!python scripts/auto_tune_rag.py \
  --profile colab-gpu \
  --dataset cuad \
  --n_records {n_records_tune} \
  --metric {metric_to_optimize} \
  --username {username}

In [ ]:
# Display the generated self-tuning execution summary report
from IPython.display import display, Markdown

tune_report_path = "reports/cuad_auto_tune_report.md"
if os.path.exists(tune_report_path):
    with open(tune_report_path, "r", encoding="utf-8") as f_tune:
        tune_content = f_tune.read()
    display(Markdown(tune_content))
else:
    print("Tuning report not found. Please run the cell above first.")

## 📊 Step 5: Run Rich Legal-RAG Diagnostics

Execute the diagnostics script to parse the run results, trace chunk/sentence boundary split rates, list LLM failure types, and output the diagnostics dashboard:

In [ ]:
# Reconstruct run_id dynamically or enter manually below:
run_id = f"{run_id_prefix}_{chunker}_{embedder}_{retriever}"

print(f"Analyzing run results for Sweep ID: {run_id}")
!python scripts/cuad_rich_analysis.py --run_id {run_id}

## 📑 Step 6: Render Diagnostic Report Dashboard

Read and display the generated diagnostic markdown report directly in the notebook session:

In [ ]:
from IPython.display import display, Markdown

report_path = "reports/cuad_diagnostic_report.md"
with open(report_path, "r", encoding="utf-8") as f:
    content = f.read()

display(Markdown(content))